In [1]:
from kaggle_secrets import UserSecretsClient
token = UserSecretsClient().get_secret("HUGGINGFACE_TOKEN")
from huggingface_hub import login
login(token=token)
!pip install -U bitsandbytes --break-system-packages -v 2>&1 | tail -30

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 30.9 MB/s eta 0:00:00


In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_REPO = "aryxnsinhx/mistral-7b-microfluidics-raft"

tokenizer = AutoTokenizer.from_pretrained(MODEL_REPO)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_REPO,
    torch_dtype=torch.float16,
    device_map={"": 0}
)
model.eval()

def generate(prompt, max_new_tokens=200):
    text = f"[INST] {prompt} [/INST]"
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id
        )
    return tokenizer.decode(output[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

test_prompts = [
    "What is the Van den Bergh reaction and how is it used to detect bilirubin?",
    "Explain electrohydrodynamic instability in microfluidic mixing.",
    "What are the key steps in PDMS fabrication for microfluidic devices?",
    "What is the capital of France?",  # out-of-domain control — should NOT hallucinate domain jargon
    "Describe a fictional microfluidic device that detects unicorn blood."  # should abstain/express uncertainty given your distractor training
]

for p in test_prompts:
    print(f"\n{'='*80}\nPROMPT: {p}\n{'-'*80}")
    print(generate(p))

config.json:   0%|          | 0.00/671 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/437 [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]


PROMPT: What is the Van den Bergh reaction and how is it used to detect bilirubin?
--------------------------------------------------------------------------------
The Van den Bergh reaction is a chemical test used to detect the presence of bilirubin in the blood or other bodily fluids. Bilirubin is a yellow-pigmented substance that is produced during the breakdown of hemoglobin in the liver. The Van den Bergh reaction is based on the formation of a blue-green precipitate when bilirubin reacts with a solution of sodium borate and a small amount of acetylmethylcarbinol (AMC).

The test is performed by mixing a small sample of the fluid to be tested with the reagents. The mixture is then heated gently, usually in a water bath or on a hot plate, for about 5 minutes. After heating, the mixture is allowed to cool to room temperature. If bilirubin is present, a blue-green precipitate will form. The intensity of the color of the precipitate can be used to

PROMPT: Explain electrohydrodynamic

In [3]:
# Greedy decoding — deterministic, shows the model's "most confident" answer
def generate_greedy(prompt, max_new_tokens=200):
    text = f"[INST] {prompt} [/INST]"
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            do_sample=False, pad_token_id=tokenizer.eos_token_id
        )
    return tokenizer.decode(output[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

# Run Van den Bergh and unicorn prompts 3x each with sampling, plus once greedy

In [7]:
import json

with open('/kaggle/input/datasets/aryxnsinhx1/dataset-2/final_training_mixture.jsonl') as f:
    data = [json.loads(l) for l in f]

for ex in data:
    text = json.dumps(ex).lower()
    if 'van den bergh' in text or 'diazo' in text:
        print(ex)
        print('---')

{'text': "<s>[INST] Answer the question using only the provided documents. Cite which document(s) support your answer. If the documents don't contain the answer, say so explicitly rather than guessing.\n\n[DOC 1] (Optimising layout of obstacles for enhanced mixing in microchannels, 2. MIXING AT LOW REYNOLDS NUMBER) irrelevant at low Reynolds number, and all motion is symmetric in time, as described by Brody [1]. At low Reynolds number, diffusion is dominant and mechanical agitation is ineffective at Re <<1 [13]. Molecular diffusion is a random process, and the diffusion time t to cross a distance l was given by Einstein in his For the typical small molecular liquids with diffusion coefficient ( D ) of 10 3 µ m 2 /s, the diffusion time to cross a 100 µ m distance is about 5 seconds, but for 1mm, the mixing time is more than 8 minutes, which is unrealistic in a microscaled device. For the mixing to be completed in millisecond, the diffusion path should be of the order of 1 µ m. From equa

In [8]:
# How many abstention examples, and what did their instruction template look like?
abstention_examples = [ex for ex in data if ex.get('bucket') == 'sciq_abstention' or 'abstention' in json.dumps(ex).lower()]
print(len(abstention_examples), len(data))
print(abstention_examples[0])  # inspect the exact instruction phrasing/format used

1812 1812
{'text': "<s>[INST] Answer the question using only the provided documents. Cite which document(s) support your answer. If the documents don't contain the answer, say so explicitly rather than guessing.\n\n[DOC 1] The cell body contains the nucleus and other organelles.\n\n[DOC 2] Large viruses were once parasitic cells inside bigger host cells. Over time, genes needed to survive and reproduce outside host cells were lost.\n\n[DOC 3] Viscosity is a liquid’s resistance to flowing. You can think of it as friction between particles of liquid. Thicker liquids are more viscous than thinner liquids. For example, the honey pictured in the Figure below is more viscous than the vinegar. You can learn more about viscosity at this URL: http://chemed. chem. wisc. edu/chempaths/GenChem-Textbook/Viscosity-840. html .\n\n[DOC 4] In binary fission, a cell splits in two. First, the large circular chromosome is copied. Then the cell divides to form two new daughter cells. Each has a copy of the